In [ ]:
import numpy as np
import pandas as pd
from math import sqrt

# ---------------------------------------------------------------------
# 1) Configuration: column order (fine-tuning end times) and baselines
# ---------------------------------------------------------------------

S_LIST = [1, 3, 5, 7, 9, 10, 20, 30, 40, 50, 70, 90, 110, 130, 150]
S_LIST = [10, 20, 30, 40, 50, 70, 90, 110, 130, 150]

# Baseline values (mean, std) and budget display taken from your ORIGINAL table.
# Keyed by T because T is unambiguous in your example and matches the original rows.
BASELINE_BY_T = {
    40: {"k": 7.98, "pct": "82.0\\%",  "baseline_mean": 98.82, "baseline_std": 0.06},
    30: {"k": 6.20, "pct": "63.67\\%", "baseline_mean": 98.66, "baseline_std": 0.07},
    20: {"k": 4.24, "pct": "43.52\\%", "baseline_mean": 98.36, "baseline_std": 0.09},
    10: {"k": 1.98, "pct": "20.38\\%", "baseline_mean": 97.42, "baseline_std": 0.08},
     9: {"k": 1.77, "pct": "18.23\\%", "baseline_mean": 97.37, "baseline_std": 0.08},  # only present in your ft=3 table
     5: {"k": 0.94, "pct": "9.69\\%",  "baseline_mean": 95.83, "baseline_std": 0.34},
     4: {"k": 0.74, "pct": "7.65\\%",  "baseline_mean": 95.03, "baseline_std": 0.40},
     3: {"k": 0.56, "pct": "5.73\\%",  "baseline_mean": 93.90, "baseline_std": 0.33},
}

# Default row orders to mimic your original tables
ROW_ORDER_FT1 = [40, 30, 20, 10, 5, 4, 3]
ROW_ORDER_FT3 = [40, 30, 20, 10, 9, 5, 4]


# ---------------------------------------------------------------------
# 2) Helpers: parsing + formatting
# ---------------------------------------------------------------------

def _to_list(x):
    """Convert cell content to a flat list of floats if possible."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, (list, tuple, np.ndarray, pd.Series)):
        # Flatten nested lists if any
        out = []
        for v in x:
            if isinstance(v, (list, tuple, np.ndarray, pd.Series)):
                out.extend(list(v))
            else:
                out.append(v)
        return [float(v) for v in out if v is not None and not (isinstance(v, float) and np.isnan(v))]
    # scalar
    return [float(x)]

def _variance_to_std(v):
    """Variance -> std. Accepts scalar or list-like."""
    vals = _to_list(v)
    if len(vals) == 0:
        return np.nan
    # If variance is provided per-row (typically one value), take mean variance then sqrt
    return float(np.sqrt(np.mean(vals)))

def _fmt_num(x, decimals=2):
    """Format a number like the table: 2 decimals, but strip trailing zeros."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    s = f"{float(x):.{decimals}f}"
    s = s.rstrip("0").rstrip(".")  # turn 98.80 -> 98.8, 99.00 -> 99
    return s

def _tightcell(mean, std, bold=False):
    """Return a LaTeX \\tightcell{mean}{std} string, optionally bolding the mean."""
    m = _fmt_num(mean, 2)
    s = _fmt_num(std, 2)
    if m == "":
        return "---"
    if bold:
        m = f"\\textbf{{{m}}}"
    # If std is empty, still keep parentheses structure consistent:
    if s == "":
        s = "0"
    return f"\\tightcell{{{m}}}{{{s}}}"

def _budget_cell(T):
    """Return budget cell \\tightcell{k}{pct} from the original table mapping."""
    if T not in BASELINE_BY_T:
        # fallback: show T only (still returns something valid)
        return f"\\tightcell{{}}{{}}"
    b = BASELINE_BY_T[T]
    return f"\\tightcell{{{_fmt_num(b['k'], 2)}}}{{{b['pct']}}}"

def _baseline_cell(T):
    """Return baseline cell \\tightcell{mean}{std} from original table."""
    if T not in BASELINE_BY_T:
        return "---"
    b = BASELINE_BY_T[T]
    return _tightcell(b["baseline_mean"], b["baseline_std"], bold=False)


# ---------------------------------------------------------------------
# 3) Core: aggregate your df -> (T, s) -> (mean, std)
# ---------------------------------------------------------------------

def aggregate_results(df):
    """
    Aggregate df into dict: agg[T][s] = (mean_acc, std_acc)
    - mean_acc comes from max_test_acc (mean over list if list)
    - std_acc comes from var_test_acc (sqrt of variance)
    If multiple rows exist per (T,s), averages mean_acc and averages variance before sqrt.
    """
    required = {"max_test_acc", "var_test_acc", "s", "T"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"DataFrame is missing required columns: {missing}")

    # group by (T, s)
    agg = {}
    for (T, s), g in df.groupby(["T", "s"], dropna=False):
        # mean accuracy: average over all provided accuracies (from lists or scalars)
        acc_vals = []
        for v in g["max_test_acc"].tolist():
            acc_vals.extend(_to_list(v))
        mean_acc = float(np.mean(acc_vals)) if len(acc_vals) else np.nan

        # std from variance: average variances (if multiple rows) then sqrt
        var_vals = []
        for v in g["var_test_acc"].tolist():
            var_vals.extend(_to_list(v))
        std_acc = float(np.sqrt(np.mean(var_vals))) if len(var_vals) else np.nan

        agg.setdefault(int(T), {})[int(s)] = (mean_acc, std_acc)

    return agg


# ---------------------------------------------------------------------
# 4) LaTeX generation: subtable + full table*
# ---------------------------------------------------------------------

def generate_subtable_latex(df, t_ft, row_order, caption, label=None, blue=False):
    """
    Generate one subtable LaTeX block.
    - blue=True reproduces your colored second subtable styling.
    """
    agg = aggregate_results(df)

    # Ensure row_order includes any T present in df but missing from the provided order
    extra_T = sorted(set(map(int, df["T"].unique())) - set(row_order))
    full_row_order = list(row_order) + extra_T

    # Build each row and compute bold best per row across s columns
    body_lines = []
    for T in full_row_order:
        # Budget + baseline cells
        budget = _budget_cell(T)
        baseline = _baseline_cell(T)

        # Collect means to decide bolding (only among S_LIST and only if present)
        means = []
        for s in S_LIST:
            if T in agg and s in agg[T]:
                means.append(agg[T][s][0])
        best = np.nanmax(means) if len(means) else np.nan

        row_cells = [budget, str(int(T)), baseline]

        for s in S_LIST:
            if T in agg and s in agg[T]:
                mean_acc, std_acc = agg[T][s]
                bold = (not np.isnan(best)) and (not np.isnan(mean_acc)) and (abs(mean_acc - best) < 1e-12)
                row_cells.append(_tightcell(mean_acc, std_acc, bold=bold))
            else:
                row_cells.append("---")

        body_lines.append(" & ".join(row_cells) + " \\\\")

    # Styling wrappers
    pre = ""
    post = ""
    caption_cmd = f"\\caption{{{caption}}}"

    if blue:
        pre = (
            "{%\n"
            "\\color{blue}%\n"
            "\\arrayrulecolor{blue}%\n"
        )
        if label:
            caption_cmd = (
                "{\\captionsetup{labelfont={color=blue}, textfont={color=blue}}%\n"
                f"{caption_cmd}\n"
                "}\n"
                f"\\label{{{label}}}"
            )
        else:
            caption_cmd = (
                "{\\captionsetup{labelfont={color=blue}, textfont={color=blue}}%\n"
                f"{caption_cmd}\n"
                "}\n"
            )
        post = "\\arrayrulecolor{black}%\n}%"

    else:
        if label:
            caption_cmd += f"\n\\label{{{label}}}"

    # Header line for s columns
    s_header = " & ".join([f"\\textbf{{{s}}}" for s in S_LIST])

    latex = f"""
\\begin{{subtable}}{{\\linewidth}}
\\centering
{caption_cmd}
\\renewcommand{{\\arraystretch}}{{1.2}}
\\setlength{{\\tabcolsep}}{{2pt}}
\\begin{{adjustbox}}{{max width=\\linewidth}}
\\begin{{tabular}}{{|c|c|c|{'c'*len(S_LIST)}|}}
\\hline
\\multirow{{2}}{{*}}{{\\shortstack{{Carbon \\\\ Budget ($k$)}}}} & \\multirow{{2}}{{*}}{{$T$}} & \\multirow{{2}}{{*}}{{Baseline}} & \\multicolumn{{{len(S_LIST)}}}{{c|}}{{\\textbf{{End time for fine-tuning ($s$)}}}} \\\\
\\cline{{4-{3+len(S_LIST)}}}
 &  &  & {s_header} \\\\
\\hline
{chr(10).join(body_lines)}
\\hline
\\end{{tabular}}
\\end{{adjustbox}}
\\end{{subtable}}
""".strip()

    if blue:
        latex = pre + "\n" + latex + "\n" + post

    return latex


def generate_full_table_latex(tmp_1, tmp_3,
                             main_caption=None,
                             label="tab:combined_table_mnist",
                             row_order_ft1=ROW_ORDER_FT1,
                             row_order_ft3=ROW_ORDER_FT3):
    """
    Generate the full LaTeX table* with two subtables (ft=1 and ft=3).
    """
    if main_caption is None:
        main_caption = (
            "Final test accuracy on the MNIST dataset for different carbon budget constraints ($k$) "
            "and fine-tuning end times ($s$). The first column reports the absolute available budget "
            "and the percentage relative to the full-budget reference. The second and third columns show "
            "the training time ($T$) and the accuracy of the slack-agnostic baseline, respectively. "
            "The remaining columns correspond to varying fine-tuning end times $s$, evaluated for two "
            "fine-tuning durations ($t_{\\mathrm{ft}} = 1$ and $t_{\\mathrm{ft}} = 3$). Numbers in parentheses "
            "denote the standard deviation across three seeds. Bold values indicate the best accuracy in each row."
        )

    sub1 = generate_subtable_latex(
        tmp_1, t_ft=1,
        row_order=row_order_ft1,
        caption="Test accuracy with fine-tuning duration $t_{\\mathrm{ft}}=1$.",
        label=None,
        blue=False
    )

    sub3 = generate_subtable_latex(
        tmp_3, t_ft=3,
        row_order=row_order_ft3,
        caption="Test accuracy with fine-tuning duration $t_{\\mathrm{ft}}=3$.",
        label="tab:carbon_budget-3ft",
        blue=True
    )

    full = f"""
\\begin{{table*}}[h!]
\\centering
\\caption{{{main_caption}}}
\\label{{{label}}}
\\scriptsize
\\renewcommand{{\\arraystretch}}{{1.05}}
\\setlength{{\\tabcolsep}}{{1pt}}

{sub1}

\\vspace{{1em}}

{sub3}

\\end{{table*}}
""".strip()

    return full


# ---------------------------------------------------------------------
# 5) Usage in notebook:
# ---------------------------------------------------------------------
# latex_table = generate_full_table_latex(tmp_1, tmp_3)
# print(latex_table)
#
# Optionally write to a .tex file:
# with open("combined_table_mnist_generated.tex", "w") as f:
#     f.write(latex_table)

In [ ]:
from plots_utils.loading import ExperimentConfig, load_experiment_results
from plots_utils.plot_availability_comparison import plot_availability_comparison
from plots_utils.plot_biased_unbiased_comparison import plot_biased_unbiased_comparison
from plots_utils.plot_av_mat import plot_av_mat
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

from matplotlib import pyplot as plt
# from plots_utils.loading import parse_tf_events_file
import numpy as np
import pandas as pd

from pathlib import Path
import os

def parse_tf_events_file(events_path, tag, time_horizon=None):
    """
    Returns the data in the file located in the folder events_paths,
    and corresponding to the tag.
    The tag can be: 'Train/Loss', 'Train/Metric', 'Test/Loss', 'Test/Metric'.
    """
    ea = EventAccumulator(events_path).Reload()
    # print(list(ea.Scalars(tag)))
    tag_values, steps = [], []
    for event in ea.Scalars(tag):
        if time_horizon is None or event.step <= time_horizon:
            tag_values.append(event.value)
            steps.append(event.step)
    return steps, tag_values

def get_exp_stats(config):
    results = list()
    for lr in config.lr_list:
        # time_horizon = time_horizons[p]  
        for algorithm in config.algorithms:
            # b_loop = config.b_values if algorithm == 'mixture' else [None] # in case we vary beta
            # for b in b_loop:
            for event in config.events:
                for seed in config.seeds:
                    for a in config.alphas:
                        for n_c in config.n_clients_list:
                            for av in config.availabilities:
                                for part in config.participations:
                                    for biased in config.biased_list:

                                        event_dir = config.get_event_dir(algorithm, lr, seed, 
                                                                            event, a, n_c, av, 
                                                                            config.n_rounds, part, biased, config.train_test)
                                        # print(event_dir)


                                        if os.path.exists(event_dir):
                                            # print('x')
                                            # _, values = parse_tf_events_file(event_dir, tag="Test/Metric", time_horizon=time_horizon)
                                            _, test_accuracy_values = parse_tf_events_file(event_dir, tag="Test/Metric")
                                            _, test_loss_values = parse_tf_events_file(event_dir, tag="Test/Loss")
                                            _, train_accuracy_values = parse_tf_events_file(event_dir, tag="Train/Metric")
                                            _, train_loss_values = parse_tf_events_file(event_dir, tag="Train/Loss")
                                            ### tag can be: 'Train/Loss', 'Train/Metric', 'Test/Loss', 'Test/Metric'
                                            max_accuracy = np.array(test_accuracy_values).max() * 100
                                            results.append({
                                                "algorithm": algorithm, 
                                                "availability": av,
                                                "alpha": a, 
                                                "participation": part,
                                                "max_test_accuracy": float(max_accuracy),
                                                "final_test_accuracy":test_accuracy_values[-1]*100,
                                                "test_accuracy": np.array(test_accuracy_values),
                                                "seed": seed,
                                                "lr": lr, "event": event, "n_clients": n_c,
                                                "biased": biased
                                            })

                                            # "b": float(b) if b else np.nan # in case we vary beta
    return pd.DataFrame(results)


main_folder = 'mnist_idle_accuracy_check'

availabilities="""
alphaF-50sl-1cb-1ft
alphaF-60sl-1cb-1ft
alphaF-70sl-1cb-1ft
alphaF-80sl-1cb-1ft
alphaF-90sl-1cb-1ft
alphaF-110sl-1cb-1ft
alphaF-130sl-1cb-1ft
alphaF-150sl-1cb-1ft
alphaF-170sl-1cb-1ft
alphaF-190sl-1cb-1ft
alphaF-40sl-2cb-1ft
alphaF-50sl-2cb-1ft
alphaF-60sl-2cb-1ft
alphaF-70sl-2cb-1ft
alphaF-80sl-2cb-1ft
alphaF-100sl-2cb-1ft
alphaF-120sl-2cb-1ft
alphaF-140sl-2cb-1ft
alphaF-160sl-2cb-1ft
alphaF-180sl-2cb-1ft
alphaF-30sl-3cb-1ft
alphaF-40sl-3cb-1ft
alphaF-50sl-3cb-1ft
alphaF-60sl-3cb-1ft
alphaF-70sl-3cb-1ft
alphaF-90sl-3cb-1ft
alphaF-110sl-3cb-1ft
alphaF-130sl-3cb-1ft
alphaF-150sl-3cb-1ft
alphaF-20sl-4cb-1ft
alphaF-30sl-4cb-1ft
alphaF-40sl-4cb-1ft
alphaF-50sl-4cb-1ft
alphaF-60sl-4cb-1ft
alphaF-80sl-4cb-1ft
alphaF-15sl-5cb-1ft
alphaF-25sl-5cb-1ft
alphaF-35sl-5cb-1ft
alphaF-45sl-5cb-1ft
alphaF-14sl-6cb-1ft
alphaF-24sl-6cb-1ft
alphaF-34sl-6cb-1ft
alphaF-13sl-7cb-1ft
alphaF-23sl-7cb-1ft
alphaF-50sl-1cb-3ft
alphaF-60sl-1cb-3ft
alphaF-70sl-1cb-3ft
alphaF-80sl-1cb-3ft
alphaF-90sl-1cb-3ft
alphaF-110sl-1cb-3ft
alphaF-130sl-1cb-3ft
alphaF-150sl-1cb-3ft
alphaF-170sl-1cb-3ft
alphaF-190sl-1cb-3ft
alphaF-40sl-2cb-3ft
alphaF-50sl-2cb-3ft
alphaF-60sl-2cb-3ft
alphaF-70sl-2cb-3ft
alphaF-80sl-2cb-3ft
alphaF-100sl-2cb-3ft
alphaF-120sl-2cb-3ft
alphaF-140sl-2cb-3ft
alphaF-160sl-2cb-3ft
alphaF-180sl-2cb-3ft
alphaF-30sl-3cb-3ft
alphaF-40sl-3cb-3ft
alphaF-50sl-3cb-3ft
alphaF-60sl-3cb-3ft
alphaF-70sl-3cb-3ft
alphaF-90sl-3cb-3ft
alphaF-110sl-3cb-3ft
alphaF-130sl-3cb-3ft
alphaF-150sl-3cb-3ft
alphaF-20sl-4cb-3ft
alphaF-30sl-4cb-3ft
alphaF-40sl-4cb-3ft
alphaF-50sl-4cb-3ft
alphaF-60sl-4cb-3ft
alphaF-80sl-4cb-3ft
alphaF-19sl-5cb-3ft
alphaF-29sl-5cb-3ft
alphaF-39sl-5cb-3ft
alphaF-15sl-6cb-3ft
alphaF-25sl-6cb-3ft
alphaF-35sl-6cb-3ft
alphaF-14sl-7cb-3ft
alphaF-24sl-7cb-3ft
""".split()


config = ExperimentConfig(base_path=os.path.join('..', 'logs'), experiment="mnist_idle0.1", seeds=["42", '78', '84'],
                          algorithms=["fedavg"], events=["global"],
                          lr_list=['5e-2', '1e-2'], alphas=["0.5"], n_clients_list=["7"],
                          availabilities=availabilities,
                          n_rounds="100", participations=["known"], biased_list=["2"], train_test="train")

results_df = get_exp_stats(config)



import ast

# Convert 'test_accuracy' column to lists if needed
results_df['test_accuracy'] = results_df['test_accuracy'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)



# # Loop over availability groups
# for availability_value, group in results_df.groupby("availability"):
    
#     plt.figure(figsize=(10, 6))
    
#     for _, row in group.iterrows():
#         label = f"lr={row['lr']} | seed={row['seed']}"
#         plt.plot(row['test_accuracy'], label=label)
    
#     plt.title(f"Test Accuracy Curves — availability = {availability_value}")
#     plt.xlabel("Epoch")
#     plt.ylabel("Test Accuracy")
#     plt.grid(True)
#     plt.legend()
#     plt.show()



In [42]:
results_df = results_df[['availability','final_test_accuracy', 'seed','lr']]
tmp = results_df.groupby(['availability']).final_test_accuracy.apply(np.vstack).to_frame().reset_index()
tmp['var_test_acc'] = tmp['final_test_accuracy'].apply(lambda x : x.var(axis=0))
tmp['max_test_acc'] = tmp['final_test_accuracy'].apply(lambda x: x.max(axis=0))
tmp = tmp[['availability', 'var_test_acc', 'max_test_acc']]
tmp["cb"] = tmp["availability"].str.extract(r"(\d+)cb").astype(int)
tmp["ft"] = tmp["availability"].str.extract(r"(\d+)ft").astype(int)
tmp["s"] = tmp["availability"].str.extract(r"(\d+)sl").astype(int)

In [43]:
cb_to_time = [40, 30, 20, 10, 5, 4, 3]

tmp_1 = tmp.loc[tmp["ft"] == 1].copy()
tmp_1=tmp_1[["availability","var_test_acc","max_test_acc", "cb", "s"]]
tmp_1["T"] = tmp_1["cb"].apply(lambda x : cb_to_time[x-1])
tmp_1["s"] = tmp_1["s"] - tmp_1["T"]
tmp_1

,availability,var_test_acc,max_test_acc,cb,s,T
0,alphaF-100sl-2cb-1ft,[0.17618077914091956],[99.08000230789185],2,70,30
2,alphaF-110sl-1cb-1ft,[0.10124695202229361],[99.04999732971191],1,70,40
4,alphaF-110sl-3cb-1ft,[0.25099011115297604],[98.47000241279602],3,90,20
6,alphaF-120sl-2cb-1ft,[0.22251418768289474],[98.79000186920166],2,90,30
8,alphaF-130sl-1cb-1ft,[0.13015494861928148],[98.91999959945679],1,90,40
10,alphaF-130sl-3cb-1ft,[25.31671485498303],[91.97999835014343],3,110,20
12,alphaF-13sl-7cb-1ft,[17.102566691973053],[96.35000228881836],7,10,3
13,alphaF-140sl-2cb-1ft,[0.30869125709721673],[98.5700011253357],2,110,30
15,alphaF-14sl-6cb-1ft,[6.728355688973132],[97.79000282287598],6,10,4
16,alphaF-150sl-1cb-1ft,[0.12445845159210951],[98.86000156402588],1,110,40


In [38]:
cb_to_time = [40, 30, 20, 10, 9, 5, 4]

tmp_3 = tmp.loc[tmp["ft"] == 3].copy()
tmp_3["T"] = tmp_3["cb"].apply(lambda x : cb_to_time[x-1])
tmp_3["s"] = tmp_3["s"] - tmp_3["T"]
tmp_3

,availability,var_test_acc,max_test_acc,cb,ft,s,T
1,alphaF-100sl-2cb-3ft,[0.16961402592367372],[98.97000193595886],2,3,70,30
3,alphaF-110sl-1cb-3ft,[0.12342212907097878],[99.14000034332275],1,3,70,40
5,alphaF-110sl-3cb-3ft,[2.8915134289025306],[97.58999943733215],3,3,90,20
7,alphaF-120sl-2cb-3ft,[0.24329296692530514],[98.79000186920166],2,3,90,30
9,alphaF-130sl-1cb-3ft,[0.19737988755331848],[99.04999732971191],1,3,90,40
11,alphaF-130sl-3cb-3ft,[16.47788709106695],[94.20999884605408],3,3,110,20
14,alphaF-140sl-2cb-3ft,[0.4697899621793529],[97.75999784469604],2,3,110,30
17,alphaF-150sl-1cb-3ft,[0.07280069640655838],[98.79000186920166],1,3,110,40
19,alphaF-150sl-3cb-3ft,[16.11765059013553],[94.48000192642212],3,3,130,20
22,alphaF-160sl-2cb-3ft,[3.2074566616399935],[91.71000123023987],2,3,130,30


In [51]:
latex_table = generate_full_table_latex(tmp_1, tmp_3)
print(latex_table)
#
# Optionally write to a .tex file:
# with open("combined_table_mnist_generated.tex", "w") as f:
#     f.write(latex_table)

\begin{table*}[h!]
\centering
\caption{Final test accuracy on the MNIST dataset for different carbon budget constraints ($k$) and fine-tuning end times ($s$). The first column reports the absolute available budget and the percentage relative to the full-budget reference. The second and third columns show the training time ($T$) and the accuracy of the slack-agnostic baseline, respectively. The remaining columns correspond to varying fine-tuning end times $s$, evaluated for two fine-tuning durations ($t_{\mathrm{ft}} = 1$ and $t_{\mathrm{ft}} = 3$). Numbers in parentheses denote the standard deviation across three seeds. Bold values indicate the best accuracy in each row.}
\label{tab:combined_table_mnist}
\scriptsize
\renewcommand{\arraystretch}{1.05}
\setlength{\tabcolsep}{1pt}

\begin{subtable}{\linewidth}
\centering
\caption{Test accuracy with fine-tuning duration $t_{\mathrm{ft}}=1$.}
\renewcommand{\arraystretch}{1.2}
\setlength{\tabcolsep}{2pt}
\begin{adjustbox}{max width=\linewidt

In [26]:
import pandas as pd
df = tmp
# --- 1) Ensure list-like numeric columns are scalars (e.g., [74.87] -> 74.87) ---
for col in ["var_test_acc", "max_test_acc"]:
    df[col] = df[col].apply(lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) > 0 else x)

# Make sure cb, sl, T are integers (optional but recommended)
df["cb"] = df["cb"].astype(int)
df["sl"] = df["sl"].astype(int)
df["T"]  = df["T"].astype(int)

# --- 2) Pivot to wide: rows are (cb, T), columns are sl, values are max_test_acc ---
wide = (df.pivot_table(index=["cb", "T"], columns="sl", values="max_test_acc", aggfunc="first")
          .sort_index()
          .sort_index(axis=1))

# Optional: nicer LaTeX column header like "sl=80" instead of just "80"
wide.columns = [f"sl={c}" for c in wide.columns]

# --- 3) Export to LaTeX ---
latex_str = wide.to_latex(
    index=True,
    na_rep="",
    float_format="%.2f",
    caption="Max test accuracy by cb, T and sl",
    label="tab:cb_T_sl",
    column_format="ll" + "r"*wide.shape[1]  # 2 left cols (cb,T) + right-aligned numeric cols
)

print(latex_str)


\begin{table}
\caption{Max test accuracy by cb, T and sl}
\label{tab:cb_T_sl}
\begin{tabular}{llrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
 &  & sl=21 & sl=30 & sl=41 & sl=50 & sl=55 & sl=60 & sl=61 & sl=65 & sl=70 & sl=75 & sl=80 & sl=81 & sl=85 & sl=90 & sl=95 & sl=100 & sl=101 & sl=105 & sl=110 & sl=115 & sl=120 & sl=121 & sl=125 & sl=130 & sl=135 & sl=140 & sl=145 & sl=150 \\
cb & T &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
1 & 40 &  &  &  &  &  &  &  &  & [74.94999766] &  &  &  &  & [74.87000227] &  &  &  &  & [75.52000284] &  &  &  &  & [75.44000149] &  &  &  & [75.55000186] \\
\cline{1-30}
2 & 30 &  &  &  &  &  &  &  & [74.83000159] &  &  &  &  & [74.91000295] &  &  &  &  & [75.01999736] &  &  &  &  & [74.98999834] &  &  &  & [75.29000044] &  \\
\cline{1-30}
3 & 20 &  &  &  &  &  & [74.29999709] &  &  &  &  & [74.87000227] &  &  &  &  & [74.58000183] &  &  &  &  & [74.81999993] &  &  &  &  & [75.2399981] &  &  \\
\cline{1-30}
4 &